# peers

> The peers folder: the envelope a message travels in, the log of all messages, and one inbox per session.

In [0]:
#| default_exp peers

In [0]:
#| hide
from nbdev.showdoc import *

In [0]:
#| hide
from fastcore.test import test_eq, ExceptionExpected

Claude Code, Codex and Ramabana sessions on one Mac can send each other notes, asks and replies. This module keeps all of it on disk, in one peers folder. It needs no network and no model, so a hook can import it and stay fast.

A message is an `Envelope`. `post` writes it to the receiver's inbox file and to the log. `take` reads the inbox and marks what it read, so two readers never get the same message. `control_call` is the old way to reach a running session; it lives here so this module needs nothing from `control`.

In [0]:
#| export
import fcntl, json, os, re, socket, subprocess, time, uuid
from dataclasses import dataclass, field, asdict, fields
from datetime import datetime, timedelta
from pathlib import Path
from fastcore.basics import ifnone

## The folder

`peers_dir` gives the peers folder. `$RAMABANA_PEERS` wins, so tests and scripts can point it elsewhere. A peer key is `f'{kind}-{session}'`; it names the peer's inbox.

In [0]:
#| export
def peers_dir(cfg=None):
    "The peers folder: `$RAMABANA_PEERS`, else `peers` in `cfg` (default `~/.config/ramabana`); made on first use."
    d = Path(os.environ.get('RAMABANA_PEERS') or Path(cfg or '~/.config/ramabana').expanduser()/'peers')
    d.mkdir(parents=True, exist_ok=True)
    return d

In [0]:
import tempfile
with tempfile.TemporaryDirectory() as t:
    test_eq(peers_dir(t), Path(t)/'peers')
    assert (Path(t)/'peers').is_dir()

## The envelope

An `Envelope` is one message. `kind` is `note`, `ask` or `reply`. `frm` says who sent it and must hold a `key` (`f'{kind}-{session}'`), because later code counts budgets by that key. `id`, `thread` and `at` fill themselves in. In `dict()` the field `frm` is written as `from`, which is a Python keyword.

In [0]:
#| export
@dataclass
class Envelope:
    "One message between peers."
    kind: str
    text: str
    frm: dict
    to: str = ''
    repo: str = ''
    thread: str = ''
    hops: int = 0
    changes: list = field(default_factory=list)
    proof: list = field(default_factory=list)
    id: str = ''
    at: float = 0.
    def __post_init__(self):
        self.id = self.id or uuid.uuid4().hex[:12]
        self.thread = self.thread or self.id
        self.at = self.at or time.time()
    def dict(self):
        "The envelope as a dict, with `frm` written as `from`."
        d = asdict(self)
        d['from'] = d.pop('frm')
        return d
    @classmethod
    def load(cls, d):
        "The envelope in `d`, a dict made by `dict`; keys that are not fields are ignored."
        d = dict(d, frm=d.get('from', d.get('frm', {})))
        return cls(**{f.name: d[f.name] for f in fields(cls) if f.name in d})

In [0]:
e = Envelope('ask', 'build Card', {'kind': 'claude', 'session': 's1', 'key': 'claude-s1'}, to='leela')
test_eq(Envelope.load(e.dict()), e)
test_eq((e.dict()['from']['key'], e.thread == e.id, len(e.id)), ('claude-s1', True, 12))

## The log and the inboxes

`log` adds one row to `log.jsonl`; `read_log` reads the rows from a time on. A row is an envelope dict or an event row such as `{'event': 'joined', 'at': ...}`. `post` writes an envelope to the receiver's inbox and to the log.

In [0]:
#| export
def _append(p, row):
    "Add `row` as one line to `p`."
    with open(p, 'a') as f: f.write(json.dumps(row) + '\n')

def _rows(p):
    "The rows in the JSON lines file `p`; none when it does not exist."
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []

def log(d, row):
    "Add `row` to the log in `d`."
    _append(Path(d)/'log.jsonl', row)

def read_log(d, since=0.):
    "The log rows in `d` made after time `since`."
    return [r for r in _rows(Path(d)/'log.jsonl') if r.get('at', 0) > since]

def post(d, key, env):
    "Put `env` in the inbox of peer `key` and in the log."
    _append(Path(d)/f'{key}.inbox.jsonl', env.dict())
    log(d, env.dict())

In [0]:
with tempfile.TemporaryDirectory() as t:
    log(t, {'event': 'joined', 'at': 5.})
    post(t, 'k', Envelope('note', 'hi', {'key': 'claude-s1'}, at=9.))
    test_eq([r.get('event') or r['text'] for r in read_log(t)], ['joined', 'hi'])
    test_eq(len(read_log(t, since=6)), 1)

## Taking messages

`take` is the only way to read an inbox. It locks the `.taken` file, reads the rows not yet taken that match `pred`, and writes their ids there. The lock means two readers, such as a hook and `tell --wait`, never take the same message. `wait_take` repeats `take` until a message comes, the time ends, or `alive()` is false.

In [0]:
#| export
def take(d, key, pred=None):
    "The inbox rows of `key` not yet taken that match `pred`, marked taken by id under a lock."
    d = Path(d)
    with open(d/f'{key}.taken', 'a+') as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        f.seek(0)
        seen = set(f.read().split())
        new = [e for e in map(Envelope.load, _rows(d/f'{key}.inbox.jsonl')) if e.id not in seen and (pred is None or pred(e))]
        f.write(''.join(f'{e.id}\n' for e in new))
    return new

def wait_take(d, key, timeout, pred=None, alive=lambda: True, poll=.5):
    "`take` until a message comes; give up after `timeout` seconds (None: never) or when `alive()` is false."
    end = None if timeout is None else time.monotonic() + timeout
    while not (got := take(d, key, pred)):
        if not alive() or (end is not None and time.monotonic() >= end): break
        time.sleep(poll)
    return got

In [0]:
with tempfile.TemporaryDirectory() as t:
    post(t, 'k', Envelope('reply', 'r', {}, thread='t1'))
    post(t, 'k', Envelope('note', 'n', {}))
    test_eq([e.text for e in take(t, 'k', lambda e: e.thread == 't1')], ['r'])
    test_eq([e.text for e in take(t, 'k')], ['n'])
    test_eq(take(t, 'k'), [])
    test_eq(wait_take(t, 'k', .1, poll=.02), [])

A dead session must not block a waiter. With `alive` false, `wait_take` returns at once, but a message posted before the call is still returned.

In [0]:
with tempfile.TemporaryDirectory() as t:
    t0 = time.monotonic()
    test_eq(wait_take(t, 'k', None, alive=lambda: False, poll=5), [])
    assert time.monotonic() - t0 < 1
    post(t, 'k', Envelope('note', 'early', {}))
    test_eq([e.text for e in wait_take(t, 'k', None, alive=lambda: False, poll=5)], ['early'])

## Registered sessions

Ramabana sessions are known to each other already. Claude Code and Codex sessions must say they exist. `register` writes one file per session, `{kind}-{session}.json`, with the folders it works in and the process id of the agent. `registered` lists the sessions that still run: it removes a file whose process is dead. A `/clear` in Claude Code gives a new session id in the same process, so `register` also removes any older file of the same `kind` and `pid`.

In [0]:
#| export
HOP_MAX, PEER_BUDGET, TOGETHER_HOURS = 6, {'messages': 40, 'turns': 20}, 12

def pid_alive(pid):
    "True when a process with id `pid` exists."
    try: os.kill(pid, 0)
    except ProcessLookupError: return False
    except (PermissionError, OverflowError): return pid > 0
    return pid > 0

def _regs(d):
    "The registration files in `d`."
    return [p for p in Path(d).glob('*.json') if not p.name.endswith('.together.json')]

def register(d, kind, session, folders, pid, name=''):
    "Write the registration of a session; any other of the same `kind` and `pid` is removed."
    for p in _regs(d):
        if (r := json.loads(p.read_text())).get('kind') == kind and r.get('pid') == pid: p.unlink()
    row = dict(kind=kind, session=session, name=name or f'{Path(folders[0]).name} · {kind}', folders=list(folders), pid=pid, key=f'{kind}-{session}', at=time.time())
    (Path(d)/f'{row["key"]}.json').write_text(json.dumps(row))
    return row

def unregister(d, kind, session):
    "Remove the registration of a session."
    (Path(d)/f'{kind}-{session}.json').unlink(missing_ok=True)

def registered(d):
    "The live registrations in `d`, oldest first; a row whose `pid` is dead is removed."
    rows = [(p, json.loads(p.read_text())) for p in _regs(d)]
    for p, r in rows:
        if not pid_alive(r['pid']): p.unlink(missing_ok=True)
    return sorted((r for _, r in rows if pid_alive(r['pid'])), key=lambda r: (r['at'], r['key']))

In [0]:
with tempfile.TemporaryDirectory() as t:
    register(t, 'claude', 's1', ['/r/abhyasa'], os.getpid())
    register(t, 'claude', 's2', ['/r/leela'], 999999)
    test_eq([(r['key'], r['name']) for r in registered(t)], [('claude-s1', 'abhyasa · claude')])
    register(t, 'claude', 's3', ['/r/abhyasa'], os.getpid())
    test_eq([r['session'] for r in registered(t)], ['s3'])
    unregister(t, 'claude', 's3')
    test_eq(registered(t), [])

A hook runs inside the agent's process tree, so it needs the id of the agent, not of the hook. `agent_pid` walks up the parents until the command is `claude` or `codex`. `git_root` gives the repository a folder belongs to, and `add_folder` lets a session claim more folders.

In [0]:
#| export
def agent_pid(names=('claude', 'codex')):
    "The nearest ancestor process whose command base name is in `names`, else the parent process."
    pid = os.getppid()
    while pid > 1:
        out = subprocess.run(['ps', '-o', 'ppid=,comm=', '-p', str(pid)], capture_output=True, text=True).stdout.split(None, 1)
        if len(out) < 2: break
        if Path(out[1].strip()).name in names: return pid
        pid = int(out[0])
    return os.getppid()

def git_root(path):
    "The git top folder of `path`, else `path` resolved."
    r = subprocess.run(['git', '-C', str(path), 'rev-parse', '--show-toplevel'], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 and r.stdout.strip() else str(Path(path).resolve())

def add_folder(d, key, path):
    "Add the folder `path` to the registration `key`; returns the new row."
    p = Path(d)/f'{key}.json'
    row = json.loads(p.read_text())
    row['folders'] = list(dict.fromkeys([*row['folders'], str(Path(path).resolve())]))
    p.write_text(json.dumps(row))
    return row

In [0]:
assert pid_alive(os.getpid()) and not pid_alive(999999)
test_eq(agent_pid(('no-such-command',)), os.getppid())
test_eq(git_root('/tmp/ramabana-no-such'), str(Path('/tmp/ramabana-no-such').resolve()))
assert Path(git_root('.')).joinpath('nbs').is_dir()
with tempfile.TemporaryDirectory() as t:
    register(t, 'codex', 'c1', ['/r/a'], os.getpid())
    test_eq(add_folder(t, 'codex-c1', '/r/b')['folders'], ['/r/a', str(Path('/r/b').resolve())])

## Work together mode

In work together mode a session may answer its partners without waiting for a person. It lasts `TOGETHER_HOURS` hours, or until a time of day such as `07:00`. `until='HH:MM'` means the next such local time after `now`, so `07:00` given at 23:00 is tomorrow. `together_state` ends an expired state and writes a `together_end` row in the log. `is_partner` says whether a sender may talk to this session: with no `partners` listed, all may.

In [0]:
#| export
def together_on(d, key, partners=(), until=None, now=None):
    "Start work together mode for `key`; `until` is 'HH:MM' (next such local time) or None (`TOGETHER_HOURS` hours)."
    now = ifnone(now, time.time())
    if until:
        h, m = map(int, until.split(':'))
        t = datetime.fromtimestamp(now).replace(hour=h, minute=m, second=0, microsecond=0)
        end = (t if t.timestamp() > now else t + timedelta(days=1)).timestamp()
    else: end = now + TOGETHER_HOURS*3600
    st = dict(started=now, until=end, partners=list(partners))
    (Path(d)/f'{key}.together.json').write_text(json.dumps(st))
    return st

def together_off(d, key):
    "End work together mode for `key`; returns the state that ended, or None."
    p = Path(d)/f'{key}.together.json'
    st = json.loads(p.read_text()) if p.exists() else None
    p.unlink(missing_ok=True)
    return st

def together_state(d, key, now=None):
    "The work together state of `key`, or None; an expired state is ended and logged."
    now = ifnone(now, time.time())
    p = Path(d)/f'{key}.together.json'
    if not p.exists(): return None
    st = json.loads(p.read_text())
    if now < st['until']: return st
    together_off(d, key)
    log(d, {'event': 'together_end', 'key': key, 'started': st['started'], 'at': now})

def is_partner(state, frm):
    "True when sender `frm` is a partner in `state`: no `partners` means all, else match by name, first word of name, or a folder name."
    if not state.get('partners'): return True
    name = frm.get('name', '')
    return bool(set(state['partners']) & {name, *name.split()[:1], *(Path(f).name for f in frm.get('folders', []))})

In [0]:
with tempfile.TemporaryDirectory() as t:
    st = together_on(t, 'k', ['leela'], now=100.)
    test_eq(together_state(t, 'k', now=200.), st)
    test_eq(together_state(t, 'k', now=13*3600), None)
    test_eq(read_log(t)[-1]['event'], 'together_end')
    test_eq(together_off(t, 'k'), None)
    together_on(t, 'k', now=1.)
    test_eq(together_off(t, 'k'), dict(started=1., until=1+12*3600., partners=[]))
test_eq([is_partner(st, {'name': n, 'folders': f}) for n, f in [('leela · claude', []), ('x', ['/r/leela']), ('abhyasa · claude', ['/r/abhyasa'])]], [True, True, False])
assert is_partner({'partners': []}, {})

## Limits

Two sessions that answer each other can loop. `limit_refusal` says why a message must not go on, or gives an empty string. A message with `hops` at `HOP_MAX` or more is refused. In work together mode, `key` may also send only `PEER_BUDGET['messages']` messages, and be asked only `PEER_BUDGET['turns']` times, since the mode started. Each refusal is written to the log.

In [0]:
#| export
def limit_refusal(d, key, env, now=None):
    "Why `env` must not go on, or ''; a refusal is logged."
    why = ''
    if env.hops >= HOP_MAX: why = f'hop limit: {env.hops} of {HOP_MAX} hops used'
    elif st := together_state(d, key, now):
        rows = read_log(d, st['started'])
        sent = sum(r.get('from', {}).get('key') == key for r in rows)
        asked = sum(r.get('to') == key and r.get('kind') == 'ask' for r in rows)
        if sent >= PEER_BUDGET['messages']: why = f"budget: {sent} of {PEER_BUDGET['messages']} messages sent"
        elif asked >= PEER_BUDGET['turns']: why = f"budget: {asked} of {PEER_BUDGET['turns']} asks received"
    if why: log(d, {'event': 'refused', 'key': key, 'why': why, 'thread': env.thread, 'at': ifnone(now, time.time())})
    return why

In [0]:
with tempfile.TemporaryDirectory() as t:
    test_eq(limit_refusal(t, 'k', Envelope('note', 'x', {})), '')
    assert 'hop' in limit_refusal(t, 'k', Envelope('reply', 'x', {}, hops=HOP_MAX))
    together_on(t, 'k')
    for i in range(20): log(t, Envelope('ask', str(i), {'key': 'o'}, to='k').dict())
    assert 'asks' in limit_refusal(t, 'k', Envelope('note', 'x', {'key': 'o'}))
    test_eq(read_log(t)[-1]['event'], 'refused')

## Targets and the folder guard

`match_peer` finds one live session from what a person types: its number in the list (from 1), its session id, its key, or the base name of one of its folders. No match, or more than one, raises `LookupError`; the second lists the matches with their numbers.

`folder_denial` guards files. A session may not change a file in a folder that another session holds, unless it holds that folder too. The path is made absolute and symlinks are followed first, so `../x` and a link cannot get round the guard.

In [0]:
#| export
def _fnames(r): return [Path(f).name for f in r.get('folders', [])]

def match_peer(rows, target):
    "The one row in `rows` that `target` names: list number (from 1), session id, key, or folder base name."
    t = str(target)
    hits = [(i, r) for i, r in enumerate(rows, 1) if t in (str(i), r.get('session'), r.get('key'), *_fnames(r))]
    if not hits: raise LookupError(f'no live session {t!r}')
    if len(hits) > 1: raise LookupError(f'more than one session matches {t!r}: ' + ', '.join(f"{i} {r['key']} ({', '.join(_fnames(r))})" for i, r in hits))
    return hits[0][1]

def folder_denial(path, me, rows):
    "Why session `me` may not change `path`, or None: another session holds a folder with it and `me` does not."
    p = (Path.cwd()/path).resolve()
    holds = lambda r: any(p.is_relative_to(Path(f).resolve()) for f in r.get('folders', []))
    others = [r for r in rows if r['key'] != me and holds(r)]
    if others and not any(holds(r) for r in rows if r['key'] == me):
        return f"{p} belongs to {others[0].get('name') or others[0]['key']}; send it a message instead"

In [0]:
rows = [{'session': 'a1', 'key': 'claude-a1', 'folders': ['/r/sanketa']}, {'session': 'b2', 'key': 'ramabana-b2', 'folders': ['/r/leela']}]
test_eq([match_peer(rows, x)['session'] for x in ['2', 'b2', 'claude-a1', 'sanketa']], ['b2', 'b2', 'a1', 'a1'])
with ExceptionExpected(LookupError, 'no live session'): match_peer(rows, 'nobody')
with ExceptionExpected(LookupError, '1 claude-a1'): match_peer(rows + [{**rows[0], 'session': 'c3', 'key': 'claude-c3'}], 'sanketa')
test_eq(folder_denial('/r/leela/x.py', 'claude-a1', rows) is None, False)
test_eq([folder_denial(p, 'ramabana-b2', rows) for p in ['/r/leela/x.py', '/elsewhere/y']], [None, None])

## Changes and proof

A reply says what changed and how the sender knows it works. In a message, a line that starts `changes:` or `proof:` holds items split on `;`. `split_proof` takes these lines out of the text and returns the body and the two lists, ready for the envelope fields `changes` and `proof`.

In [0]:
#| export
def split_proof(text):
    "Split `text` into the body and the items of its `changes:` and `proof:` lines."
    body, items = [], dict(changes=[], proof=[])
    for l in text.splitlines():
        if m := re.match(r'\s*(changes|proof)\s*:(.*)', l, re.I): items[m[1].lower()] += [s.strip() for s in m[2].split(';') if s.strip()]
        else: body.append(l)
    return '\n'.join(body).strip(), items['changes'], items['proof']

In [0]:
test_eq(split_proof('Done.\nchanges: a.py; abc123\nProof: pytest -q → 12 passed'), ('Done.', ['a.py', 'abc123'], ['pytest -q → 12 passed']))
test_eq(split_proof('Only text.'), ('Only text.', [], []))

## The control socket call

`control_call(path, method, **params)` sends one request to a session's control socket and returns the `result`. An error reply raises `RuntimeError`. It moved here from `control`, which still exports it.

In [0]:
#| export
def control_call(path, method, timeout=30, **params):
    "Send one request to the `Control` socket at `path` and return its `result`; an error reply raises `RuntimeError`."
    with socket.socket(socket.AF_UNIX) as s:
        s.settimeout(timeout)
        s.connect(str(path))
        s.sendall((json.dumps({'id': 1, 'method': method, 'params': params}) + '\n').encode())
        out = json.loads(s.makefile('rb').readline())
    if 'error' in out: raise RuntimeError(out['error']['message'])
    return out['result']

`control_call` needs a live session, so `control` tests it with a real socket. Here a check that it exists and fails with an `OSError` when nothing listens.

In [0]:
with ExceptionExpected(OSError): control_call('/tmp/ramabana-no-such.sock', 'status', timeout=1)